# 05.04 · Build your own MCP server with FastMCP

**Module 05 · MCP & External Tools** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 150–180 min |
| 📋 Prerequisites | Lessons 05.02 and 05.03. |
| 🔑 Keys | `GROQ_API_KEY` (optional) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** a FastMCP server with tools, a resource template and a prompt.
2. **Design** tools a model can use well: clear names and docstrings, typed arguments, annotations, small structured results, helpful errors.
3. **Test** a server in-memory (like a unit test), then from a LangChain agent.
4. **Run** it as a process (stdio and Streamable HTTP) and connect it to an IDE (VS Code / Cursor) on Windows.
5. **Read** the course's `notes_server.py` and explain its design choices.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [FastMCP: Quickstart](https://gofastmcp.com/getting-started/quickstart) | The smallest server |
| [FastMCP: Tools](https://gofastmcp.com/servers/tools) | Types, annotations, errors (`ToolError`, `mask_error_details`), structured output |
| [FastMCP: Resources](https://gofastmcp.com/servers/resources) · [Prompts](https://gofastmcp.com/servers/prompts) | Templates and prompt functions |
| [FastMCP: Running your server](https://gofastmcp.com/deployment/running-server) | `run(transport="stdio" / "http")` |
| [MCP: Build a server](https://modelcontextprotocol.io/docs/develop/build-server) | The protocol's own tutorial |
| [VS Code: MCP servers](https://code.visualstudio.com/docs/copilot/customization/mcp-servers) · [Cursor: MCP](https://cursor.com/docs/context/mcp) | IDE configuration (`mcp.json`) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langchain[mcp]>=1.4.4,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "fastmcp", "mcp",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/05-mcp-and-external-tools/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import logging
import warnings

# (a) Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# (b) `langchain.mcp` is beta and says so once per process (LangChainBetaWarning). Read it once in 05.02; muted after.
from langchain_core._api import LangChainBetaWarning  # noqa: E402  (import first: langchain_core adds its own filter)

warnings.filterwarnings("ignore", message=".*langchain.mcp.* is in beta", category=LangChainBetaWarning)
# (c) In-process FastMCP servers log every failing tool call with a long boxed traceback. The client still gets
#     the error as a ToolMessage(status="error"); we show it ourselves. Remove these lines to see the server's view.
import fastmcp  # noqa: E402,F401  (importing fastmcp configures its logger, so set the level after the import)

logging.getLogger("fastmcp").setLevel(logging.CRITICAL)

In [ ]:
# Step 5: Imports.
import inspect
import json
import subprocess
import sys
import tempfile
from typing import Annotated

from fastmcp import Client, FastMCP
from fastmcp.exceptions import ToolError
from langchain.agents import create_agent
from langchain.mcp import MCPAdapter
from langchain.messages import AIMessage, HumanMessage
from pydantic import Field

from mastery.mcp import serve_http, tool_info
from mastery.mcp_servers import notes_server
from mastery.testing import ScriptedChatModel

print("FastMCP ready")

**You should now see** `FastMCP ready`.

---
## Part 1 · A first server: a course glossary

A FastMCP server is a Python object; decorators turn functions into MCP primitives. The function's **name, docstring and type hints become the tool's name, description and JSON schema**: that's what the model reads, so write them for the model.

In [ ]:
# 🧪 Offline: a three-primitive server in ~30 lines.
GLOSSARY = {
    "mcp": "Model Context Protocol: an open standard for connecting AI apps to tools and data.",
    "checkpointer": "Saves graph state per thread so an agent can remember and resume.",
    "middleware": "Hooks around model and tool calls in create_agent (Module 04).",
}

glossary = FastMCP("course-glossary", instructions="Definitions of course terms. Read-only.")


@glossary.tool(annotations={"readOnlyHint": True, "idempotentHint": True, "openWorldHint": False})
def define(term: Annotated[str, Field(description="One term, e.g. 'checkpointer'.", max_length=40)]) -> dict:
    """Define a course term. Returns the term and a one-sentence definition."""
    key = term.strip().lower()
    if key not in GLOSSARY:
        # ToolError text is always shown to the client: say what to do next.
        raise ToolError(f"Unknown term {term!r}. Known terms: {', '.join(sorted(GLOSSARY))}.")
    return {"term": key, "definition": GLOSSARY[key]}


@glossary.resource("glossary://terms", mime_type="application/json")
def all_terms() -> str:
    """Every term the glossary knows."""
    return json.dumps(sorted(GLOSSARY))


@glossary.prompt
def explain_like_a_tutor(term: str) -> str:
    """Ask for a beginner-friendly explanation of one term."""
    return f"Use the define tool for '{term}', then explain it to a beginner with one example."


async with Client(glossary) as client:
    tools = await client.list_tools()
    print("tool  :", tools[0].name, "|", tools[0].description)
    print("schema:", json.dumps(tools[0].input_schema["properties"]))
    print("call  :", (await client.call_tool("define", {"term": "MCP"})).structured_content)
    print("res   :", (await client.read_resource("glossary://terms"))[0].text)
    bad = await client.call_tool("define", {"term": "graph"}, raise_on_error=False)
    print("error :", bad.content[0].text)

**You should now see** the tool name and docstring, a schema with `description` and `maxLength: 40`, `call : {'term': 'mcp', 'definition': ...}`, the resource `["checkpointer", "mcp", "middleware"]`, and `error : Unknown term 'graph'. Known terms: ...` (a `ToolError` message goes to the client exactly as written).

### What just happened?

- `def define(term: Annotated[str, Field(...)]) -> dict` → an input schema **and** an output schema; the returned dict is sent as `structuredContent` (plus a JSON text copy for older clients).
- `annotations` are *hints* for the client (read-only, idempotent, no outside world). They don't enforce anything on the server.
- `raise ToolError(...)` → an `isError` result whose text the model can act on.

---
## Part 2 · Errors: what the client should (not) see

An unexpected exception can leak internals (paths, SQL, secrets in messages). `FastMCP(..., mask_error_details=True)` hides the text of ordinary exceptions; `ToolError` messages still go through, because you wrote them for the client.

In [ ]:
# 🧪 Offline: the same bug with and without masking.
def make(mask: bool) -> FastMCP:
    server = FastMCP("leaky", mask_error_details=mask)

    @server.tool
    def lookup(user_id: int) -> str:
        """Look up a user."""
        raise RuntimeError("psycopg: password authentication failed for user 'admin' at db.internal:5432")

    return server

for mask in (False, True):
    async with Client(make(mask)) as client:
        result = await client.call_tool("lookup", {"user_id": 1}, raise_on_error=False)
        print(f"mask={mask!s:<5} →", result.content[0].text)

**You should now see** the database detail with `mask=False`, and only `Error calling tool 'lookup'` with `mask=True`. Use `mask_error_details=True` for any server others connect to, and raise `ToolError` for messages you *want* the model to see.

---
## Part 3 · The course server: `src/mastery/mcp_servers/notes_server.py`

The project uses this server. It's built by a function, `build_server(notes_dir, tokens=None)`, so tests and notebooks get a fresh server on a temporary folder.

| Design choice | Why |
|---|---|
| `save_note`, `search_notes`, `list_notes`, `delete_note`, `lesson_info` | Small, specific tools beat one `notes(action=...)` tool: clearer schemas, per-tool permissions |
| Annotations on every tool (`delete_note` is `destructiveHint=True`) | The client (05.05) can ask a human before side effects |
| Limits: title 120 chars, body 4,000, 5 tags; `search_notes(limit≤20)` | Models (and attackers) send huge inputs; results must fit in context |
| `ValueError` with a clear message on bad input | Becomes an `isError` result the model can correct |
| Atomic write (`.tmp` + replace), UTF-8 | No half-written file on a crash; `ñ`/`é` safe on Windows |
| One JSON file per **owner** | With auth (05.05) each user only sees their own notes |
| `notes://{note_id}` resource, `study_plan` prompt | Shows all three primitives in one server |

In [ ]:
# 🧪 Offline: read the real source of one tool (what the model sees is the docstring + signature).
source = inspect.getsource(notes_server.build_server)
start = source.index("    def save_note")
print(source[start:source.index("\n\n", start + 200)])

**You should now see** the `save_note` function: its typed arguments, its docstring written for a model, the validation, and the returned `{id, title, tags}`.

In [ ]:
# 🧪 Offline: test the server the way tests/unit/test_mcp.py does (in-memory, temporary folder).
server = notes_server.build_server(tempfile.mkdtemp(prefix="m05-test-"))
async with Client(server) as client:
    saved = (await client.call_tool("save_note", {"title": "Señal de éxito", "body": "Notas en UTF-8: ñ, é, ü.", "tags": ["Windows"]})).data
    hits = (await client.call_tool("search_notes", {"query": "utf-8 windows"})).structured_content["result"]
    too_long = await client.call_tool("save_note", {"title": "x" * 500, "body": "b"}, raise_on_error=False)
assert hits and hits[0]["id"] == saved["id"], hits
print("saved :", saved)
print("search:", [(h["title"], h["score"]) for h in hits])
print("limits:", "error" if too_long.is_error else "trimmed to", len(too_long.data["title"]) if not too_long.is_error else too_long.content[0].text)
print("✅ in-memory tests passed")

**You should now see** the saved note (accents intact), the search hit, how the title limit is enforced, and `✅ in-memory tests passed`. The same checks, plus HTTP and auth, run in `pytest tests/unit/test_mcp.py`.

---
## Part 4 · Use it from an agent

In [ ]:
# 🧪 Offline: a scripted agent saves and then finds a note through MCPAdapter (in-memory).
async with MCPAdapter(notes_server.build_server(tempfile.mkdtemp(prefix="m05-agent-"))) as adapter:
    tools = await adapter.list_tools()

script = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "save_note", "id": "a1", "args": {"title": "MCP transports", "body": "stdio for IDEs, HTTP for shared servers.", "tags": ["mcp"]}}]),
    AIMessage("", tool_calls=[{"name": "search_notes", "id": "a2", "args": {"query": "transports"}}]),
    "Saved, and I can find it again: 'MCP transports'.",
])
agent = create_agent(script, tools=tools)
out = await agent.ainvoke({"messages": [HumanMessage("Save a note about MCP transports, then check it's there.")]})
for m in out["messages"]:
    if m.type == "tool":
        print(f"[{m.name}]", m.artifact["structured_content"])
print(out["messages"][-1].text)

**You should now see** `[save_note] {'id': 'n-...', ...}`, `[search_notes] {'result': [{... 'title': 'MCP transports' ...}]}`, and the final sentence.

> 💸 Uses a few hundred to a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: a real model chooses the notes tools itself.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    safe_tools = [t for t in tools if t.name != "delete_note"]   # no destructive tool without HITL (05.05)
    live_agent = create_agent(get_model("fast", temperature=0), tools=safe_tools,
                              system_prompt="You manage the user's study notes with the notes tools. Be brief.")
    out = await live_agent.ainvoke({"messages": [HumanMessage(
        "Save a note titled 'Top-level await' saying 'await in Jupyter, asyncio.run in scripts', tag it async. Then list my notes.")]})
    print([m.name for m in out["messages"] if m.type == "tool"])
    print(out["messages"][-1].text[:400])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) `['save_note', 'list_notes']` (or `search_notes`) and a short confirmation listing both notes. Here the agent wrote without asking anyone (we left `delete_note` out on purpose); 05.05 adds the human approval.

---
## Part 5 · Run it as a process

From the repo root in Git Bash (env active). `mastery` is installed in editable mode (`pip install -e .`), so `-m` works from any folder:

```bash
# stdio (an IDE or client starts this for you; run by hand it just waits for JSON-RPC on stdin, Ctrl+C to stop)
python -m mastery.mcp_servers.notes_server

# Streamable HTTP on http://127.0.0.1:8770/mcp
python -m mastery.mcp_servers.notes_server --transport http --port 8770
```

Notes are saved under `data/outputs/m05/notes/` (git-ignored); change it with `--notes-dir` or `MASTERY_NOTES_DIR`.

In [ ]:
# 🧪 Offline: the CLI really starts: --help, then an HTTP run via serve_http (which uses the same command).
shown = subprocess.run([sys.executable, "-m", "mastery.mcp_servers.notes_server", "--help"],
                       capture_output=True, text=True, encoding="utf-8", timeout=60)
print(shown.stdout.splitlines()[0])
with serve_http(env={"MASTERY_NOTES_DIR": tempfile.mkdtemp(prefix="m05-http-")}) as url:
    async with MCPAdapter(url) as adapter:
        print(url, "→", [row["name"] for row in tool_info(await adapter.list_tools())])

**You should now see** the `usage: ...` line and the URL with the five tool names.

### Connect it to your IDE (VS Code or Cursor)

Both read a **portable** `.mcp.json` at the workspace root (top-level `mcpServers`). VS Code also reads `.vscode/mcp.json` (top-level `servers`); Cursor reads `.cursor/mcp.json`. Use the **full path to your env's Python**, otherwise the IDE starts a different Python that doesn't have `mastery` installed. The next cell writes a ready-to-copy file with *your* path.

In [ ]:
# 🧪 Offline: write an IDE config for this machine (into data/outputs, not your repo root: you copy it yourself).
ide_config = {"mcpServers": {"course-notes": {
    "command": sys.executable,                       # e.g. C:\\Users\\you\\anaconda3\\envs\\lc-mastery\\python.exe
    "args": ["-m", "mastery.mcp_servers.notes_server"],
    "env": {"MASTERY_NOTES_DIR": str(REPO_ROOT / "data" / "outputs" / "m05" / "notes"), "PYTHONIOENCODING": "utf-8"},
}}}
target = REPO_ROOT / "data" / "outputs" / "m05" / "mcp.json"
target.parent.mkdir(parents=True, exist_ok=True)
target.write_text(json.dumps(ide_config, indent=2), encoding="utf-8")
print(target.read_text(encoding="utf-8"))
print("\nCopy to:", REPO_ROOT / ".mcp.json", "(VS Code + Cursor) — or merge into .cursor/mcp.json")

**You should now see** a JSON config whose `command` is your env's `python.exe` (on Windows, with doubled backslashes: that's correct JSON), and where to copy it. Then:

- **VS Code:** open the repo, run **MCP: List Servers** → `course-notes` → *Start*; in Copilot Chat (Agent mode) ask *"search my notes for transports"*. VS Code asks before each tool call.
- **Cursor:** *Settings → MCP* shows `course-notes` with a green dot; ask in Agent chat.
- **Debug by hand (optional, needs Node.js):** `npx @modelcontextprotocol/inspector`, transport *STDIO*, command = the same `python.exe`, arguments `-m mastery.mcp_servers.notes_server`.

`.mcp.json` contains a local path: it's personal. Don't commit it (the course `.gitignore` already ignores `/.mcp.json`).

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| IDE shows the server red, log says `No module named mastery` | IDE started another Python (`python` on PATH, or the Windows Store alias) | Full path to the env's `python.exe` in `command` |
| IDE log: `can't open file ... notes_server.py` | Relative path in `args` | Use `-m mastery.mcp_servers.notes_server` (no path at all) |
| `Invalid \escape` when the IDE reads `mcp.json` | Single backslashes in a hand-written Windows path | Double them (`C:\\Users\\...`) or use forward slashes (`C:/Users/...`) |
| Tool missing in the client | Exception at import or decorator time in the server | Run `python -m mastery.mcp_servers.notes_server --help` in Git Bash to see the error |
| Client hangs / garbage on stdio | `print()` in a stdio server | Log to stderr |
| `Address already in use` on 8770 | Another copy is running | Stop it, or `--port 8771` |
| `UnicodeEncodeError` on Windows | Console code page | `PYTHONIOENCODING=utf-8` (in the config above) |
| `npx` not found | Node.js not installed | The Inspector is optional; install Node LTS or skip it |

## ⚠️ Production notes

> ⚠️ **Production note: an MCP tool is an API.**
> Validate every argument, cap sizes, return small structured results, version your server, and keep the tool names stable: models and clients depend on them. `mask_error_details=True` for any server others connect to.

> ⚠️ **Production note: local servers run with your rights.**
> A stdio server can read anything you can. Give it only the folders and environment variables it needs (`env` in the config), and prefer a dedicated data folder like `MASTERY_NOTES_DIR`.

## 🏋️ Exercises

1. **(Easy)** Add `"thread"` to `GLOSSARY` and call `define` again. Do you need to rebuild the server?
2. **(Medium)** Add a `rename_note(note_id, title)` tool to a copy of the notes server. Which annotations should it have?
3. **(Medium)** Add a `glossary://terms/{term}` resource template to the glossary server.
4. **(Stretch)** Connect `course-notes` to your IDE and ask it to save a note. Then open the JSON file under `data/outputs/m05/notes/` and find the note.

<details>
<summary><b>Show solutions</b></summary>

**1.** No: `define` reads `GLOSSARY` at call time. (Changing the *function signature* or docstring would need a new server.)

**2.** `readOnlyHint=False`, `destructiveHint=False` (it changes but doesn't delete), `idempotentHint=True` (same rename twice = same result), `openWorldHint=False`; and `auth=require_scopes("notes:write")` like `save_note`.

**3.** `@glossary.resource("glossary://terms/{term}")` + `def one_term(term: str) -> str: return GLOSSARY[term.lower()]`; read with `await client.read_resource("glossary://terms/mcp")`.

**4.** The file is `local.json` (owner `local`, no auth): a list of notes with `id`, `title`, `body`, `tags`, `created`.
</details>

## ✅ Checkpoint

- [ ] I can build a FastMCP server with a tool, a resource and a prompt.
- [ ] I write tool docstrings, types, limits and errors for the model.
- [ ] I know when to use `ToolError` and `mask_error_details=True`.
- [ ] I can test a server in-memory and from an agent.
- [ ] I can run it over stdio and HTTP and connect it to an IDE on Windows.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Check the [FastMCP changelog](https://gofastmcp.com/changelog) for changes since 4.1.0 to tool annotations, `mask_error_details` or `run(transport=...)`, and whether VS Code still recommends the portable `.mcp.json`. Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- FastMCP turns typed, documented functions into tools, resources and prompts.
- Design tools for the model: specific names, clear docstrings, limits, small structured results, actionable errors.
- Test in-memory first, then from an agent, then as a process.
- IDEs start stdio servers from `mcp.json`; on Windows use the env's full `python.exe` path.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 05.05 · MCP auth and security** → [`05-05-mcp-auth-and-security.ipynb`](05-05-mcp-auth-and-security.ipynb).

[Module 05 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)